# X5_Q1 — Tabla maestra + variables de estado del activo

Primer notebook de **X5_alt_v2** (evolución paralela de X5_alt — ver
[`docs/plans/X5_alternativo.md`](../docs/plans/X5_alternativo.md) sección 10,
documento rector [`docs/plans/x5_contexto_conversacion.md`](../docs/plans/x5_contexto_conversacion.md)).
No reemplaza `X5_P1.ipynb`/`X5_P2.ipynb` (v1, congelados en `scripts/otros_strings/`).

Este notebook construye, para cada activo de `valores`, la misma tabla
continua en el tiempo (una fila por vela H1) que ya resolvía `X5_P1.ipynb` —
precio + X3 (técnicos) + X2 (fundamental) — y la deja como punto de partida
para las variables de estado del activo (drawdown, volatilidad, ruido,
tendencia/régimen, liquidez) que se agregan en los ítems siguientes de
`X5_alt_v2` (TODOs en `docs/tracking/todos.md`, sección X5_alt_v2).

**Input único que se puede cambiar — lista, no string** (a diferencia del
`valor` singular de v1): soporta varios activos sin reescribir esta celda.

In [1]:
valores = ['BTCUSD']  # lista de activos; default solo BTCUSD (ver plan X5_alternativo.md §10.2)

## 0. Preparación

Reutilizamos `_calcular_todos_indicadores` de `X3_technical_features.py`
(igual que `X5_P1.ipynb`) en vez de reimplementar los indicadores técnicos.

In [2]:
import json  # para leer x2_history.json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# cwd = scripts/ al correr en Jupyter; fallback a __file__ por si se ejecuta como script
SCRIPTS_DIR = Path.cwd() if (Path.cwd() / 'config.py').exists() else Path(__file__).resolve().parent
sys.path.insert(0, str(SCRIPTS_DIR))
import config as cfg  # noqa: E402
from X3_technical_features import _calcular_todos_indicadores  # noqa: E402  (reusa X3 en vez de reimplementar indicadores)

pd.set_option('display.width', 140)  # evita que pandas trunque columnas al imprimir DataFrames anchos

print(f"Activos: {valores}")

Activos: ['BTCUSD']


## 1. Tabla base por activo (precio + X3 + X2)

Misma lógica de ensamblado que `X5_P1.ipynb` (precio OHLCV + indicadores
técnicos de X3 + fundamental de X2 con forward-fill), encapsulada en una
función para poder correrla sobre cada activo de `valores`.

In [3]:
def _cargar_tabla_base(valor):
    # Precio: el CSV no viene ordenado por fecha y puede traer velas duplicadas (ver X5_P1 §1)
    path_precio = cfg.CARPETA_DATA / f'{valor}.csv'
    df_precio = pd.read_csv(path_precio)
    df_precio['DateTime'] = pd.to_datetime(df_precio['DateTime'])
    df_precio = (df_precio.sort_values('DateTime')
                           .drop_duplicates(subset=['DateTime'])
                           .reset_index(drop=True))

    # X3 sobre el histórico completo (no incremental) para que indicadores con
    # warm-up largo (EMA, RSI, SMA 200) sean correctos desde la primera vela útil.
    # conjunto_N vacío = sin distancia a soportes (fuera de v0, igual que X5_P1 §2).
    df_x3 = _calcular_todos_indicadores(df_precio, set())
    df_x3 = df_x3.drop(columns=['dist_nearest_support', 'dist_floor_support', 'density_2pct'])
    df_x3 = df_x3.rename(columns={c: f'x3_{c}' for c in df_x3.columns if c != 'datetime'})

    # X2: x2_history.json trae todos los activos mezclados — filtramos por {valor}
    path_x2 = cfg.CARPETA_FUNDAMENTALS / 'x2_history.json'
    with open(path_x2) as f:
        x2_hist = json.load(f)
    registros = [e for e in x2_hist if e.get('activo') == valor]
    df_x2 = pd.DataFrame([
        {
            'DateTime': pd.to_datetime(e.get('periodo_inicio') or e['date']),
            'x2_score': e.get('score'),
            'x2_score_cross': e.get('score_cross'),
            'x2_score_tendencia': e.get('score_tendencia'),
        }
        for e in registros
    ]).sort_values('DateTime').reset_index(drop=True)  # merge_asof exige orden ascendente

    # Precio y X3 comparten índice/orden → join exacto por DateTime
    tabla = df_precio.merge(df_x3, left_on='DateTime', right_on='datetime').drop(columns=['datetime'])
    # merge_asof(direction='backward') = forward-fill del último score de X2 conocido
    tabla = pd.merge_asof(tabla, df_x2, on='DateTime', direction='backward')
    return tabla


tablas = {valor: _cargar_tabla_base(valor) for valor in valores}

for valor, tabla in tablas.items():
    print(f"{valor}: {tabla.shape[0]} filas x {tabla.shape[1]} columnas — {tabla['DateTime'].min()} → {tabla['DateTime'].max()}")

BTCUSD: 40283 filas x 37 columnas — 2021-10-27 16:00:00 → 2026-06-02 21:00:00
